<a href="https://colab.research.google.com/github/UPEI-CS/CS1910-Students/blob/main/lectures/10-Problem-Solving-Debugging/CS1910_Problem-Solving-Debugging-Parity_Handout.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CS1910 Computer Science 1: Problem Solving**
# **Bit Parity**

## Save your own copy first

You will probably want your own copy of this workbook, so that you can keep your answers and come back to review it. **Do this before you start**, because anything you type into the original is not saved. Choose:

***File->Save a copy in Drive***

or

***Copy to Drive*** if it appears in the above menu.

Google sign-in is required. You will then be able to open the copy from your Google Drive account later (Colab keeps it in the *Colab Notebooks* folder).

If you can't use Google Drive, choose ***File->Download .ipynb*** and keep the file by email (signing in to a cloud email service) or on a USB key. To open it in another Colab session later, you would have to move it into your Google Drive account.

Remember that you can always open the original version from the course Moodle. It just won't contain anything you have typed into its cells.

## How to use this notebook

This is a **try-at-home** problem. Work on your own, at your own pace. It takes about 60 minutes.

This problem is about the **process** of problem solving, not just the final code. Work through the four stages in order: **Understand → Design → Implement → Evaluate**. Resist the urge to jump straight to code!

1. **Run the setup cell(s)** (▶️ below) first.
2. Answer the questions in the ✏️ cells (double-click a ✏️ cell to edit it).
3. When a code cell ends with checks, keep working until every check shows ✅.

If you are stuck for more than 10 minutes, write down what you tried and move on, then bring your question to office hours or the help centre.

## What this notebook practises

A *bit* has two states, which we write as `0` and `1`. Computers add a *parity bit* to data so that the receiver can detect some transmission errors. You will work through all four problem-solving stages: pinning down the problem, breaking it into small reusable functions, writing them, and then testing them, including finding out what kinds of errors parity can and cannot catch.

### Key points to take away

- A **bit string** contains only `0`s and `1`s, for example `"0100110"`. Its *parity* is even if it contains an even number of `1`s.
- To give a bit string even parity, add `"0"` to the end if it already has an even number of `1`s, and `"1"` if it has an odd number. The added bit is the count of `1`s **% 2**.
- Bit strings are **strings**: you add a bit with string concatenation (`bits + "0"`), not arithmetic.
- **Decomposition** into small functions (`count_ones`, `has_even_parity`, `add_parity_bit`) makes each piece easy to test and reuse.
- **Edge cases** matter: the empty string has zero `1`s, which is even, so it becomes `"0"`.
- Parity detects any **single** flipped bit, but **two** flipped bits cancel out and go undetected.

In [ ]:
#@title ▶️ Run this cell first
#@markdown This cell sets up two helpers, `check` and `check_case`, that tell you
#@markdown whether your answer is right. You do not need to understand how they work.
import math as _math

_cases_passed = {}


def _same(got, expected):
    if isinstance(expected, float) and isinstance(got, (int, float)) and not isinstance(got, bool):
        return _math.isclose(got, expected, rel_tol=1e-9, abs_tol=1e-9)
    return type(got) == type(expected) and got == expected


def check(got, expected, label="Answer"):
    if _same(got, expected):
        print(f"✅ {label}: {got!r}")
    elif got == expected:
        print(f"❌ {label}: the value looks right, but it is a {type(got).__name__}"
              f" and should be a {type(expected).__name__}")
    else:
        print(f"❌ {label}: got {got!r}, expected {expected!r}")


def check_case(name, case, got, expected_by_case):
    if case not in expected_by_case:
        print(f"ℹ️ {case!r} is not one of the test cases for {name}. Try one of: "
              + ", ".join(repr(c) for c in expected_by_case))
        return
    passed = _cases_passed.setdefault(name, set())
    check(got, expected_by_case[case], f"{name} with {case!r}")
    if _same(got, expected_by_case[case]):
        passed.add(case)
    still = [repr(c) for c in expected_by_case if c not in passed]
    if still:
        print(f"   {len(passed)} of {len(expected_by_case)} test cases passed. "
              f"Change the input and run again. Still to try: {', '.join(still)}")
    else:
        print(f"   🎉 All {len(expected_by_case)} test cases passed!")


print("✅ Practice checker ready.")

## Stage 1: Understand (about 10 minutes)


Here is the problem as it was given:

> In computer science a bit is something that has two states (on and off). We represent bits with `0` or `1`. A bit string is a string containing only `0`s and `1`s, for example `"0100110"`.
>
> *Parity* is about the number of `1`s in a bit string: if there is an even number, we say the bit string has **even parity**. Suppose we always want our bit strings to have even parity. Given a bit string, add one character to the end so that the new bit string has even parity.
>
> Example: `"101"` becomes `"1010"`.

Answer these before you write any code.

1. What is the input, and what is its type? What is the output, and what is its type?
2. In your own words, what does "even parity" mean?
3. Is the output the bit that is added, or the whole new string? How can you tell?
4. What should happen for the empty string `""`?
5. What should happen if the input contains something other than `0` and `1`, such as `"102"`? Does the problem say?
6. How could the problem be generalized?

✏️ **Your answers** (double-click to edit):

1. 
2. 
3. 
4. 
5. 
6. 

### Work examples by hand

In [ ]:
# 1.1 Work these out by hand. Write the whole new string each time.
answer_101 = ""        # "101"
answer_111 = ""        # "111"
answer_0100110 = ""    # "0100110"
answer_empty = ""      # ""  (the empty string)

# ---- Checks: run the cell to test your answers (do not change below) ----
check(answer_101, "1010", 'for "101"')
check(answer_111, "1111", 'for "111"')
check(answer_0100110, "01001101", 'for "0100110"')
check(answer_empty, "0", 'for ""')

### Predict, then run

Each cell is independent. Write your prediction first, then run it. These are the building blocks you will need.

In [ ]:
# 1.2. Predict the output, then run the cell.
# My prediction:

print("101" + "0")

In [ ]:
# 1.3. Predict the output, then run the cell.
# My prediction:

print("101" + 0)

In [ ]:
# 1.4. Predict the output, then run the cell.
# My prediction:

print(3 % 2, 4 % 2)

In [ ]:
# 1.5. Predict the output, then run the cell.
# My prediction:

print(str(3 % 2))

In [ ]:
# 1.6. Predict the output, then run the cell.
# My prediction:

print("0100110".count("1"))

## Stage 2: Design (about 10 minutes)


**Decompose** the problem into small functions that each do one job, and can be reused:

- `count_ones(bits)` returns the number of `"1"` characters in `bits`.
- `has_even_parity(bits)` returns `True` if `bits` has an even number of `1`s.
- `add_parity_bit(bits)` returns `bits` with the correct parity bit added to the end.

Pattern recognition: look back at your hand-worked answers. Is there a simple rule connecting the number of `1`s and the bit that was added?

Write pseudocode for all three functions. Which function should call which?

✏️ **Your plan** (double-click to edit; write it before you code):

1. 
2. 
3. 

## Stage 3: Implement (about 15 minutes)

Write the three functions, using your plan. Write `count_ones` with a **loop** (not the `count` method), then **reuse** your functions: `has_even_parity` should call `count_ones`, and `add_parity_bit` should call one of the others.

In [ ]:
def count_ones(bits):
    # Your code here (use a loop)
    pass


def has_even_parity(bits):
    # Your code here (call count_ones)
    pass


def add_parity_bit(bits):
    # Your code here (reuse your other functions)
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(count_ones("0100110"), 3, 'count_ones("0100110")')
check(count_ones(""), 0, 'count_ones("")')
check(has_even_parity("101"), True, 'has_even_parity("101")')
check(has_even_parity("111"), False, 'has_even_parity("111")')
check(has_even_parity(""), True, 'has_even_parity("")')
check(add_parity_bit("101"), "1010", 'add_parity_bit("101")')
check(add_parity_bit("111"), "1111", 'add_parity_bit("111")')
check(add_parity_bit("0100110"), "01001101", 'add_parity_bit("0100110")')
check(add_parity_bit("0000"), "00000", 'add_parity_bit("0000")')
check(add_parity_bit("1"), "11", 'add_parity_bit("1")')
check(add_parity_bit(""), "0", 'add_parity_bit("")')

## Stage 4: Evaluate (about 15 minutes)


Parity bits exist to **detect errors**. The sender adds a parity bit; the receiver checks that what arrived still has even parity. If a bit was flipped (a `0` became a `1` or the other way round) on the way, the parity changes.

First write `flip_bit(bits, i)`, which returns `bits` with the character at index `i` flipped. *Hint:* use slicing: everything before `i`, the flipped character, then everything after `i`.

In [ ]:
def flip_bit(bits, i):
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(flip_bit("1010", 0), "0010", 'flip_bit("1010", 0)')
check(flip_bit("1010", 3), "1011", 'flip_bit("1010", 3)')
check(flip_bit("1", 0), "0", 'flip_bit("1", 0)')

In [ ]:
# 4.2 Send a message, damage it, and check it.
sent = add_parity_bit("1101001")
print("sent:              ", sent, has_even_parity(sent))

one_error = flip_bit(sent, 2)
print("one bit flipped:   ", one_error, has_even_parity(one_error))

two_errors = flip_bit(flip_bit(sent, 2), 5)
print("two bits flipped:  ", two_errors, has_even_parity(two_errors))

Run the cell, then answer:

1. When one bit was flipped, did the receiver detect the error? Why?
2. When two bits were flipped, was the error detected? Explain why not.
3. Does it matter *which* single bit is flipped, including the parity bit itself? Try a few.
4. Can the receiver tell *which* bit was wrong? What could it do about the error?
5. List the test cases you used in Stage 3 and say which are normal and which are edge cases. Is there any case you think is missing?

✏️ **Your answers** (double-click to edit):

1. 
2. 
3. 
4. 
5. 

### Extensions

Try these if you have time.

In [ ]:
# ★ Generalize: add a parameter even=True. When even is False, add the bit that gives ODD parity.
def add_parity_bit(bits, even=True):
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(add_parity_bit("101"), "1010", 'add_parity_bit("101")')
check(add_parity_bit("101", even=False), "1011", 'add_parity_bit("101", even=False)')
check(add_parity_bit("", even=False), "1", 'add_parity_bit("", even=False)')

In [ ]:
# ★★ Robustness: write is_bit_string(text), which returns True only if every character is "0" or "1".
def is_bit_string(text):
    # Your code here
    pass

# ---- Checks: run the cell to test your code (do not change below) ----
check(is_bit_string("0110"), True, 'is_bit_string("0110")')
check(is_bit_string(""), True, 'is_bit_string("")')
check(is_bit_string("102"), False, 'is_bit_string("102")')
check(is_bit_string("1 0"), False, 'is_bit_string("1 0")')

## Before you leave

Tick off each statement you can honestly agree with. For any you can't, go back to the matching part of this lab, the lecture workbook, or the book chapter.

- [ ] I can explain what even parity is, and add a parity bit to any bit string, including the empty string.
- [ ] I broke the problem into small functions and reused them.
- [ ] I can explain which errors a parity bit detects and which it misses.
- [ ] I chose normal and edge test cases before trusting my code.

✏️ **Reflection** (double-click to edit; answer together with your partner):

1. Which stage of the process took you longest? Was it the one you expected?
2. How did splitting the problem into three small functions help you test it?
3. Where else might a computer check data for errors?

*Your answers:*

1. 
2. 
3. 